In [1]:
print("Hello World")

Hello World


In [25]:
import sys
from pathlib import Path
cwd = Path.cwd()
for candidate in (cwd, cwd.parent):
    if (candidate / "helpers" / "common.py").exists() and str(candidate) not in sys.path:
        sys.path.append(str(candidate))
        break


from helpers.common import tavily_search, together_ai_llm

## Inbuilt Tool

In [3]:
tavily_search.invoke({"query": "What happened at the last wimbledon"})

{'query': 'What happened at the last wimbledon',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://www.atptour.com/en/news/wimbledon-2025-results',
   'title': 'What were the Wimbledon results? | ATP Tour | Tennis',
   'content': "# ATP Tour. #### TOURNAMENT RESULTS. #### PLAYER RESULTS. ## What were the Wimbledon results? Jannik Sinner sank his great rival Carlos Alcaraz on Sunday evening at Wimbledon to become the first Italian to lift a singles trophy at the grass major. **Read More from Wimbledon:**  Sinner gains Alcaraz revenge, wins first Wimbledon crown. Sinner claws closer to Alcaraz in Big Titles chase with Wimbledon triumph. ### Read More News View All News. * {{player.ranking}} Rank {{opponent.ranking}}. player.Age : '-'}} Age {{opponent.Age ? * {{getWeightInLB.player}} Weight {{getWeightInLB.opponent}}. * {{getHeightInFeetInch.player}} Height {{getHeightInFeetInch.opponent}}. * {{getPlayHand.player}} Plays {{getPlayHand.opponent}}. 

In [15]:
from langchain_core.tools import tool


@tool
def web_search(query):
  """
  Search the web for realtime and latest information.
  for examples, news, stock market, weather updates etc.
    
  Args:
  query: The search query
  """
  response = tavily_search.invoke({'query': query})
  return response

web_search.invoke({
  'query': "What happened at the last wimbledon"
})


{'query': 'What happened at the last wimbledon',
 'response_time': 0.64,
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://www.atptour.com/en/news/wimbledon-2025-results',
   'title': 'What were the Wimbledon results? | ATP Tour | Tennis',
   'content': "# ATP Tour. #### TOURNAMENT RESULTS. #### PLAYER RESULTS. ## What were the Wimbledon results? Jannik Sinner sank his great rival Carlos Alcaraz on Sunday evening at Wimbledon to become the first Italian to lift a singles trophy at the grass major. **Read More from Wimbledon:**  Sinner gains Alcaraz revenge, wins first Wimbledon crown. Sinner claws closer to Alcaraz in Big Titles chase with Wimbledon triumph. ### Read More News View All News. * {{player.ranking}} Rank {{opponent.ranking}}. player.Age : '-'}} Age {{opponent.Age ? * {{getWeightInLB.player}} Weight {{getWeightInLB.opponent}}. * {{getHeightInFeetInch.player}} Height {{getHeightInFeetInch.opponent}}. * {{getPlayHand.player}} Plays {{

## Custom Tools (Add, Multiply)

In [16]:
@tool
def add(a:int, b:int) -> int:
  """
  Add two integer numbers together
    
  Args:
  a: First Integer
  b: Second Integer
  """

  return int(a) + int(b)

@tool
def multiply(a:int, b:int) -> int:
  """
  Multiply two integer numbers together
    
  Args:
  a: First Integer
  b: Second Integer
  """
  return int(a) * int(b)

In [17]:
all_tools = [web_search, add, multiply]
all_tools

[StructuredTool(name='web_search', description='Search the web for realtime and latest information.\nfor examples, news, stock market, weather updates etc.\n\nArgs:\nquery: The search query', args_schema=<class 'langchain_core.utils.pydantic.web_search'>, func=<function web_search at 0x740170bed120>),
 StructuredTool(name='add', description='Add two integer numbers together\n\nArgs:\na: First Integer\nb: Second Integer', args_schema=<class 'langchain_core.utils.pydantic.add'>, func=<function add at 0x74016cd05080>),
 StructuredTool(name='multiply', description='Multiply two integer numbers together\n\nArgs:\na: First Integer\nb: Second Integer', args_schema=<class 'langchain_core.utils.pydantic.multiply'>, func=<function multiply at 0x74016cd06520>)]

In [28]:
llm_with_tools = together_ai_llm.bind_tools(all_tools)
# llm_with_tools